# 🛡️ CkCk Hoax Detection AI — Training Notebook

**Track B: The Privacy Brain (NLP / Generative AI)**

Fine-tuning IndoBERT-base-p2 for Indonesian hoax detection with integrated PII filtering.

---

**Dataset**: 4 source CSVs (TurnBackHoax, Antaranews, Detik, Kompas) with `judul` + `clean_text` concatenation.

## 1. Setup & Imports

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
import yaml

# Add project root to path
sys.path.insert(0, os.path.abspath('.'))

from src.utils import set_seed, get_device, count_parameters, load_config
from src.dataset import create_dataloaders, load_multi_csv
from src.preprocessing import TextPreprocessor
from src.pii_filter import PIIFilter
from src.trainer import Trainer

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Load configuration
config = load_config('config.yaml')
set_seed(config['training']['seed'])
device = get_device()

print(f"\nModel: {config['model']['name']}")
print(f"Epochs: {config['training']['epochs']}")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Learning rate: {config['training']['learning_rate']}")

## 2. Data Exploration

In [ ]:
# Load all training CSVs individually for exploration
data_cfg = config['data']
train_dir = data_cfg['train_dir']
train_files = data_cfg['train_files']

dfs = {}
for filename in train_files:
    path = os.path.join(train_dir, filename)
    df = pd.read_csv(path)
    source_name = filename.replace('Cleaned_', '').replace('.csv', '')
    dfs[source_name] = df
    print(f'{source_name}: {len(df)} rows | Labels: {df["label"].value_counts().to_dict()}')

# Combine for overall stats
combined_df = pd.concat(dfs.values(), ignore_index=True)
print(f'\n📊 Total combined: {len(combined_df)} samples')
print(f'Label distribution:\n{combined_df["label"].value_counts()}')

In [ ]:
# Per-source breakdown chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Samples per source
source_counts = {name: len(df) for name, df in dfs.items()}
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
axes[0].bar(source_counts.keys(), source_counts.values(), color=colors)
axes[0].set_title('Samples per Source')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

# Overall label distribution
combined_df['label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c']
)
axes[1].set_title('Overall Label Distribution')
axes[1].set_xticklabels(['Valid (0)', 'Hoax (1)'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis (judul + clean_text combined)
combined_df['combined_text'] = combined_df['judul'].astype(str) + '. ' + combined_df['clean_text'].astype(str)
combined_df['text_length'] = combined_df['combined_text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Text length by label
for label, color, name in [(0, '#2ecc71', 'Valid'), (1, '#e74c3c', 'Hoax')]:
    subset = combined_df[combined_df['label'] == label]
    axes[0].hist(subset['text_length'], bins=50, alpha=0.6, color=color, label=name)
axes[0].set_title('Text Length Distribution by Label')
axes[0].set_xlabel('Character count (judul + clean_text)')
axes[0].legend()

# Title length
combined_df['title_length'] = combined_df['judul'].str.len()
combined_df['title_length'].hist(bins=50, ax=axes[1], color='#9b59b6', alpha=0.7)
axes[1].set_title('Title (judul) Length Distribution')
axes[1].set_xlabel('Character count')

plt.tight_layout()
plt.show()

print(f'Combined text length — Mean: {combined_df["text_length"].mean():.0f}, '
      f'Median: {combined_df["text_length"].median():.0f}, '
      f'Max: {combined_df["text_length"].max()}')

In [ ]:
# Preview samples from each source
for name, df in dfs.items():
    print(f'\n--- {name} (label={df["label"].iloc[0]}) ---')
    sample = df.iloc[0]
    print(f'  Judul: {sample["judul"][:100]}...')
    print(f'  Clean text: {sample["clean_text"][:120]}...')

## 3. PII Filter Verification

In [ ]:
# Test PII Filter
pii_filter = PIIFilter()

test_texts = [
    'NIK saya 3201234506780001 tolong dijaga.',
    'Hubungi di +6281234567890 atau email budi@gmail.com',
    'Transfer ke rekening 1234567890123456',
    'Berita ini tidak mengandung data pribadi.',
]

for text in test_texts:
    result = pii_filter.filter(text)
    print(f'Input:    {result["original_text"]}')
    print(f'Filtered: {result["filtered_text"]}')
    print(f'PII found: {result["pii_count"]}')
    print()

## 4. Preprocessing Test

In [ ]:
# Test preprocessing pipeline
preprocessor = TextPreprocessor(use_stemmer=False)

samples = [
    'BREAKING!!! Vaksin COVID berbahaya!! https://hoax.com #antivax',
    '<p>Pemerintah <b>mengumumkan</b> kebijakan baru.</p>',
    'Gak percaya gw dgn berita ini!!!',
]

for text in samples:
    cleaned = preprocessor.clean(text, normalize_slang=True)
    print(f'Original: {text}')
    print(f'Cleaned:  {cleaned}')
    print()

## 5. Model Setup & Parameter Count

In [ ]:
# Load model and verify parameter count (Constraint B-1)
model = AutoModelForSequenceClassification.from_pretrained(
    config['model']['name'],
    num_labels=config['model']['num_labels'],
)

total_params = count_parameters(model)
print(f'\n✅ Constraint B-1: {total_params:,} params (limit: 4,000,000,000)')
print(f'   Using {total_params/4_000_000_000*100:.2f}% of allowed budget')

## 6. Training

In [ ]:
# Initialize trainer and run training
trainer = Trainer.from_config('config.yaml')
trainer.setup()
history = trainer.train()

## 7. Training Curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-o', label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history['val_accuracy'], 'g-o')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(epochs, history['val_f1'], 'm-o')
axes[2].set_title('Validation F1 Score')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluation on Test Set

In [ ]:
# Evaluate on test set
test_metrics = trainer.evaluate()

if test_metrics:
    print(f"\n📊 Final Test Metrics:")
    print(f"   Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"   F1 Score:  {test_metrics['f1']:.4f}")
    print(f"   Precision: {test_metrics['precision']:.4f}")
    print(f"   Recall:    {test_metrics['recall']:.4f}")

## 9. Save Final Model

In [ ]:
# Model is automatically saved during training (best checkpoint)
# Verify saved model exists
model_path = os.path.join(config['paths']['model_dir'], 'best_model')
if os.path.exists(model_path):
    files = os.listdir(model_path)
    print(f'✅ Model saved at: {model_path}')
    print(f'   Files: {files}')
else:
    print('⚠️  No saved model found. Training may not have completed.')

---

**Training complete!** Proceed to `inference.ipynb` for the clean inference pipeline.